In [2]:
import csv
import json
import re
from decimal import Decimal
from pathlib import Path

import pdfplumber


PDF_PATH = Path(r"C:\Users\N_R_KHAN\Documents\ITL_PROJECTS\Proofreading_central_planning\VS\input_files\4502859819.pdf")

# These are extracted-data files, not Python source files.
JSON_OUTPUT = Path(r"C:\Users\N_R_KHAN\Documents\ITL_PROJECTS\Proofreading_central_planning\VS\output_files\4502859819_extracted.json")
CSV_OUTPUT = Path(r"C:\Users\N_R_KHAN\Documents\ITL_PROJECTS\Proofreading_central_planning\VS\output_files\4502859819_order_lines.csv")

# Change to False if you only want the result printed.
SAVE_OUTPUT = True


ITEM_PATTERN = re.compile(
    r"^(?P<item_code>\d{10})\s+"
    r"(?P<item_description>.+?)\s+"
    r"(?P<supplier_reference>\d{6}/[A-Z0-9]+)\s*"
    r"(?:/\s*)?"
    r"(?P<item_quantity>[\d,]+\.\d{3})\s+"
    r"(?P<uom>[A-Z]+)\s+"
    r"(?P<item_value>[\d,]+\.\d{2})\s+"
    r"\+\s*(?P<tolerance_plus>\d+(?:\.\d+)?)\s*"
    r"/-\s*(?P<tolerance_minus>\d+(?:\.\d+)?)$"
)

SALES_ORDER_PATTERN = re.compile(
    r"^(?P<sales_order>\d+)\s*/\s*(?P<sales_order_line>\d+)$"
)

SIZE_ROW_PATTERN = re.compile(
    r"^(?P<po_line>\d+)\s+"
    r"(?P<size>[A-Z0-9-]+)\s+"
    r"(?P<quantity>[\d,]+\.\d{3})\s+"
    r"(?P<uom>[A-Z]+)\s+"
    r"(?P<price>[\d,]+\.\d+)\s*/\s*"
    r"(?P<price_per>[\d,]+)\s+"
    r"(?P<ship_mode>.+)$"
)


def first_match(pattern, text, default=None, flags=re.MULTILINE):
    match = re.search(pattern, text, flags)
    return match.group(1).strip() if match else default


def decimal_value(value):
    """Convert a formatted number such as 10,743.000 to Decimal."""
    return Decimal(value.replace(",", ""))


def clean_line(line):
    return re.sub(r"\s+", " ", line).strip()


def is_repeated_header_or_footer(line):
    return (
        line.startswith("PO Number -")
        or line.startswith("PO SALES ORDER/")
        or line.startswith("LINE# UNIT SURCHARGE")
        or line.startswith("Acceptance of this order")
    )


def extract_pdf_pages(pdf_path):
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text(
                x_tolerance=2,
                y_tolerance=3,
            ) or ""

            # Layout text is useful for visually checking multi-column headers.
            layout_text = page.extract_text(
                layout=True,
                x_density=7.25,
                y_density=13,
            ) or ""

            pages.append(
                {
                    "page_number": page_number,
                    "text": text,
                    "layout_text": layout_text,
                    "word_count": len(page.extract_words()),
                    "image_count": len(page.images),
                }
            )

    return pages


def extract_header(first_page_text):
    """Extract important purchase-order header fields."""
    return {
        "po_number": first_match(
            r"\bPO Number\s*-?\s*(\d+)",
            first_page_text,
        ),
        "total_value": first_match(
            r"\bTotal Value\s+([\d,]+\.\d{2})",
            first_page_text,
        ),
        "currency": first_match(
            r"\bCurrency\s+([A-Z]{3})",
            first_page_text,
        ),
        "po_date": first_match(
            r"^Date\s+(\d{2}\.\d{2}\.\d{4})",
            first_page_text,
        ),
        "last_revised": first_match(
            r"\bLast Revised\s+(\d{2}\.\d{2}\.\d{4})",
            first_page_text,
        ),
        "version": first_match(
            r"\bVersion\s+(\d+)",
            first_page_text,
        ),
        "payment_term": first_match(
            r"^Payment Term\s*:\s*(.+)$",
            first_page_text,
        ),
        "inco_term": first_match(
            r"^Inco Term\s*:\s*(.+)$",
            first_page_text,
        ),
        "end_buyer": first_match(
            r"\bEnd Buyer\s*:\s*(\S+)",
            first_page_text,
        ),
        "contact": first_match(
            r"\bContact\s+(\S+)",
            first_page_text,
        ),
        "email": first_match(
            r"\bEmail\s+([\w.+-]+@[\w.-]+\.[A-Za-z]{2,})",
            first_page_text,
        ),
        "supplier_name": "INTERNATIONAL TRIMMINGS & LABELS BANGLADESH",
        "notify_party": "MAS Intimates Bangladesh (Pvt) Ltd.",
    }


def extract_items(pages):
    """
    Parse product items, sales orders and size rows.

    State is deliberately retained between pages because several items and
    sales orders continue onto the following page.
    """
    items = []
    current_item = None
    current_sales_order = None
    inside_table = False
    unparsed_table_lines = []

    # Pages 1-5 contain the purchase-order rows.
    for page in pages[:5]:
        for raw_line in page["text"].splitlines():
            line = clean_line(raw_line)

            if not line:
                continue

            if line.startswith("PO SALES ORDER/"):
                inside_table = True
                continue

            if is_repeated_header_or_footer(line):
                continue

            item_match = ITEM_PATTERN.match(line)
            if item_match:
                values = item_match.groupdict()

                current_item = {
                    "item_code": values["item_code"],
                    "item_description": values["item_description"],
                    "description_lines": [],
                    "supplier_reference": values["supplier_reference"],
                    "item_quantity": values["item_quantity"],
                    "uom": values["uom"],
                    "item_value": values["item_value"],
                    "tolerance": {
                        "plus_percent": values["tolerance_plus"],
                        "minus_percent": values["tolerance_minus"],
                    },
                    "sales_orders": {},
                }

                items.append(current_item)
                current_sales_order = None
                continue

            sales_order_match = SALES_ORDER_PATTERN.match(line)
            if sales_order_match and current_item:
                sales_order = sales_order_match.group("sales_order")
                sales_order_line = sales_order_match.group("sales_order_line")
                current_sales_order = f"{sales_order}/{sales_order_line}"

                current_item["sales_orders"].setdefault(
                    current_sales_order,
                    {
                        "sales_order": sales_order,
                        "sales_order_line": sales_order_line,
                        "size_rows": [],
                    },
                )
                continue

            size_match = SIZE_ROW_PATTERN.match(line)
            if size_match and current_item:
                values = size_match.groupdict()

                row = {
                    "po_line": int(values["po_line"]),
                    "size": values["size"],
                    "quantity": values["quantity"],
                    "uom": values["uom"],
                    "price": values["price"],
                    "price_per": values["price_per"].replace(",", ""),
                    "ship_mode": values["ship_mode"],
                    "source_page": page["page_number"],
                }

                if current_sales_order is None:
                    # This indicates an unexpected document layout.
                    row["warning"] = "No preceding sales order was detected"
                    current_item.setdefault("orphan_size_rows", []).append(row)
                else:
                    current_item["sales_orders"][current_sales_order][
                        "size_rows"
                    ].append(row)

                continue

            # Second description line, e.g. "LB 07655 PPK-1023 C1".
            if current_item and re.match(r"^/?\s*LB\s+\d+", line):
                current_item["description_lines"].append(
                    line.lstrip("/").strip()
                )
                continue

            if inside_table:
                unparsed_table_lines.append(
                    {
                        "page": page["page_number"],
                        "text": line,
                    }
                )

    return items, unparsed_table_lines


def validate_items(items):
    validations = []

    for item in items:
        rows = [
            row
            for sales_order in item["sales_orders"].values()
            for row in sales_order["size_rows"]
        ]

        extracted_total = sum(
            (decimal_value(row["quantity"]) for row in rows),
            Decimal("0"),
        )
        stated_total = decimal_value(item["item_quantity"])

        validations.append(
            {
                "item_code": item["item_code"],
                "number_of_size_rows": len(rows),
                "stated_quantity": f"{stated_total:.3f}",
                "calculated_quantity": f"{extracted_total:.3f}",
                "quantity_matches": extracted_total == stated_total,
            }
        )

    return validations


def extract_remarks_and_terms(last_page_text):
    remarks = ""
    terms_text = ""

    if "REMARKS" in last_page_text:
        after_remarks = last_page_text.split("REMARKS", 1)[1]

        if "TERMS AND CONDITIONS" in after_remarks:
            remarks, terms_text = after_remarks.split(
                "TERMS AND CONDITIONS",
                1,
            )
        else:
            remarks = after_remarks

    remarks = remarks.strip()
    terms_text = terms_text.strip()

    # Split numbered terms while also retaining the complete original text.
    terms = []
    for section in re.split(r"\n(?=\d+\.\s)", terms_text):
        section = section.strip()
        if not section:
            continue

        match = re.match(r"(?P<number>\d+)\.\s*(?P<text>.*)", section, re.DOTALL)
        if match:
            terms.append(
                {
                    "number": int(match.group("number")),
                    "text": clean_line(match.group("text")),
                }
            )
        else:
            terms.append(
                {
                    "number": None,
                    "text": clean_line(section),
                }
            )

    return {
        "remarks_raw": remarks,
        "terms_raw": terms_text,
        "numbered_terms": terms,
    }


def build_csv_rows(items):
    rows = []

    for item in items:
        for sales_order in item["sales_orders"].values():
            for size_row in sales_order["size_rows"]:
                rows.append(
                    {
                        "item_code": item["item_code"],
                        "item_description": item["item_description"],
                        "product_description": " ".join(
                            item["description_lines"]
                        ),
                        "supplier_reference": item["supplier_reference"],
                        "item_quantity": item["item_quantity"],
                        "item_value": item["item_value"],
                        "sales_order": sales_order["sales_order"],
                        "sales_order_line": sales_order["sales_order_line"],
                        **size_row,
                    }
                )

    return rows


def extract_purchase_order(pdf_path):
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF was not found: {pdf_path.resolve()}")

    pages = extract_pdf_pages(pdf_path)

    # A born-digital PDF should contain substantial embedded text.
    character_count = sum(len(page["text"].strip()) for page in pages)
    if character_count < 500:
        raise ValueError(
            "Very little embedded text was detected. "
            "This document may require OCR."
        )

    header = extract_header(pages[0]["text"])
    items, unparsed_lines = extract_items(pages)
    validations = validate_items(items)
    final_sections = extract_remarks_and_terms(pages[-1]["text"])

    calculated_po_value = sum(
        (decimal_value(item["item_value"]) for item in items),
        Decimal("0"),
    )

    stated_po_value = (
        decimal_value(header["total_value"])
        if header["total_value"]
        else None
    )

    return {
        "source_file": str(pdf_path),
        "document_type": "Purchase Order",
        "extraction_method": "embedded text + deterministic state-machine parser",
        "page_count": len(pages),
        "header": header,
        "items": items,
        "remarks_and_terms": final_sections,
        "validation": {
            "items": validations,
            "all_item_quantities_match": all(
                result["quantity_matches"] for result in validations
            ),
            "stated_po_value": (
                f"{stated_po_value:.2f}" if stated_po_value is not None else None
            ),
            "calculated_po_value": f"{calculated_po_value:.2f}",
            "po_value_matches": (
                calculated_po_value == stated_po_value
                if stated_po_value is not None
                else None
            ),
            "unparsed_table_lines": unparsed_lines,
        },
        # Retained so every extracted character can be audited.
        "raw_pages": pages,
    }


data = extract_purchase_order(PDF_PATH)

# Display the structured result.
print(json.dumps(data, indent=2, ensure_ascii=False))

if SAVE_OUTPUT:
    JSON_OUTPUT.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    csv_rows = build_csv_rows(data["items"])
    if csv_rows:
        with CSV_OUTPUT.open(
            "w",
            newline="",
            encoding="utf-8-sig",
        ) as csv_file:
            writer = csv.DictWriter(
                csv_file,
                fieldnames=csv_rows[0].keys(),
            )
            writer.writeheader()
            writer.writerows(csv_rows)

    print(f"\nJSON saved to: {JSON_OUTPUT.resolve()}")
    print(f"CSV saved to:  {CSV_OUTPUT.resolve()}")

{
  "source_file": "C:\\Users\\N_R_KHAN\\Documents\\ITL_PROJECTS\\Proofreading_central_planning\\VS\\input_files\\4502859819.pdf",
  "document_type": "Purchase Order",
  "extraction_method": "embedded text + deterministic state-machine parser",
  "page_count": 6,
  "header": {
    "po_number": "4502859819",
    "total_value": "1,671.25",
    "currency": "USD",
    "po_date": "14.05.2026",
    "last_revised": "14.05.2026",
    "version": "1",
    "payment_term": "Within 90 days Due net",
    "inco_term": "Ex Works/Ex Works",
    "end_buyer": "Pink",
    "contact": "MALIKASI",
    "email": "malikasi@masholdings.com",
    "supplier_name": "INTERNATIONAL TRIMMINGS & LABELS BANGLADESH",
    "notify_party": "MAS Intimates Bangladesh (Pvt) Ltd."
  },
  "items": [
    {
      "item_code": "2001431649",
      "item_description": "LBL LB07655 PPK1023 C1 MWW001",
      "description_lines": [
        "LB 07655 PPK-1023 C1"
      ],
      "supplier_reference": "416213/SPT",
      "item_quantity": "

In [31]:
import pdfplumber

pdf_path = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/4502859819.pdf")

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()

        # print(f"\n{'='*50}")
        # print(f"Page {page_number}")
        # print(f"{'='*50}")

        if text:
            print(text)
        else:
            print("No text found.")

Attn : Notify Party/Deliver To :
Purchase Order
INTERNATIONAL TRIMMINGS & MAS Intimates Bangladesh (Pvt) Ltd.
LABELSBAN Karnaphuli epz
North-Patenga
9th Floor. Prominent Tower, Plot # 01, Road - 02, PO Number 4502859819
Sector-03, Total Value 1,671.25 Chittagong 4204
1230Dhaka
Currency USD
Bangladesh Bangladesh
Date 14.05.2026 Chittagong - A100
Tel : 880258956329 Last Revised 14.05.2026
Version 1 Bill To: Consignee :
MAS Intimates Bangladesh (Pvt) Ltd.
Payment Term : Within 90 days Due net
FSFB 1
Inco Term : Ex Works/Ex Works
Karnaphuli epz
End Buyer : Pink Contact MALIKASI North-Patenga
Email malikasi@masholdings.com Chittagong
36014990 Tel Bangladesh
Tax Code-
2026/7 Fax
PO SALES ORDER/ LINE ITEM CODE ITEM DESCRIPTION/ SUPPLIER REF SIZE SIZE QTY UOM PRICE/PERDISCOUNT/ EX-FACT TOLERANCE
LINE# UNIT SURCHARGE DATE SHIP MODE
2001431649 LBL LB07655 PPK1023 C1 MWW001 416213/SPT / 10,743.000 PC 334.25 + 2.0 /- 2.0
LB 07655 PPK-1023 C1
1001667108 / 10
190 L 126.000 PC 31.11 / 1,000 Truck
190

## PO, Factory ID, month code extraction  

In [27]:
import re
from itertools import combinations
import pdfplumber


# PDF_PATH = r"../input_files/4502859819.pdf"
# PDF_PATH = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502903587.pdf")
# PDF_PATH = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502903592.pdf")
# PDF_PATH = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502914965.pdf")
# PDF_PATH = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502914995.pdf")
# PDF_PATH = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502915150.pdf")
PDF_PATH = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502916826.pdf")



def find_boxes(page, tolerance=4):
    horizontal = [
        e for e in page.edges
        if e["orientation"] == "h" and e["width"] > 120
    ]
    vertical = [
        e for e in page.edges
        if e["orientation"] == "v" and e["height"] > 80
    ]

    boxes = set()

    for first, second in combinations(horizontal, 2):
        top, bottom = sorted((first, second), key=lambda e: e["top"])

        # Prevent zero-height or very small boxes.
        if bottom["top"] - top["top"] < 20:
            continue

        if (
            abs(top["x0"] - bottom["x0"]) > tolerance
            or abs(top["x1"] - bottom["x1"]) > tolerance
        ):
            continue

        left_exists = any(
            abs(v["x0"] - top["x0"]) <= tolerance
            and v["top"] <= top["top"] + tolerance
            and v["bottom"] >= bottom["top"] - tolerance
            for v in vertical
        )

        right_exists = any(
            abs(v["x0"] - top["x1"]) <= tolerance
            and v["top"] <= top["top"] + tolerance
            and v["bottom"] >= bottom["top"] - tolerance
            for v in vertical
        )

        if left_exists and right_exists:
            box = (
                round(top["x0"], 2),
                round(top["top"], 2),
                round(top["x1"], 2),
                round(bottom["top"], 2),
            )
            boxes.add(box)

    return list(boxes)


with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]

    box_texts = [
        page.crop(box).extract_text() or ""
        for box in find_boxes(page)
    ]

# Find the Purchase Order box.
po_box = next(
    text for text in box_texts
    if "Purchase Order" in text and "PO Number" in text
)

# Find the Payment Term box.
payment_box = next(
    text for text in box_texts
    if "Payment Term" in text
)

po_match = re.search(
    r"PO\s*Number\s*(\d{10})",
    po_box,
    re.IGNORECASE,
)

factory_match = re.search(
    r"\b\d{8}\b",
    payment_box,
)

date_match = re.search(
    r"\b\d{1,4}[/-]\d{1,4}\b",
    payment_box,
)

po_number = po_match.group(1) if po_match else None
factory_id = factory_match.group(0) if factory_match else None
date_of_mfr = date_match.group(0) if date_match else None

result = {
    "po_number": po_number,
    "factory_id": factory_id,
    "date_of_mfr": date_of_mfr,
}

print(result)

{'po_number': '4502916826', 'factory_id': None, 'date_of_mfr': None}


## PO line, so number, item code, description, size, quantity extraction

In [28]:
import re
import pdfplumber
from pathlib import Path

# pdf_path = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/4502859819.pdf")
# pdf_path = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502903587.pdf")
# pdf_path = Path(r"C:/Users/N_R_KHAN/Documents/ITL_PROJECTS/Proofreading_central_planning/VS/input_files/po/4502903592.pdf")


all_text = ""

with pdfplumber.open(PDF_PATH) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            all_text += page_text + "\n"

m = re.search(r"LINE#.*?\n(.*?)\nREMARKS", all_text, re.S)

if m:
    extracted = m.group(1).strip()
else:
    raise ValueError("No LINE# to REMARKS section found.")

lines = extracted.splitlines()

po_numbers = []

for line in lines:
    po_match = re.search(
        r"PO\s*Number\s*-\s*(\d{10})",
        line,
        re.IGNORECASE,
    )
    if po_match:
        po_numbers.append(po_match.group(1))

if po_numbers and len(set(po_numbers)) == 1:
    po_number = po_numbers[0]
    print("PO number is same on all pages:", po_number)
else:
    raise ValueError(f"PO number is not same for all pages: {po_numbers}")

remove_starts = (
    "Acceptance of this order",
    "PO SALES ORDER",
    "LINE#",
)

clean_lines = [
    line for line in lines
    if line.strip()
    and not line.lstrip().startswith(remove_starts)
    and not re.search(r"PO\s*Number\s*-\s*\d{10}", line, re.IGNORECASE)
]

clean_text = "\n".join(clean_lines)
print(clean_text)

PO number is same on all pages: 4502916826
3000863694 TKT LB 7689 11275526 7YBM / LB 7689 5.000 PC 0.05 + 2.0 /- 2.0
1001692776 / 20
90 L 2.000 PC 11.34 / 1,000 Truck
90 S 3.000 PC 11.34 / 1,000 Truck
3000863695 TKT LB 7689 11291868 7ZAZ / LB 7689 14.000 PC 0.14 + 2.0 /- 2.0
1001692775 / 10
50 L 2.000 PC 11.34 / 1,000 Truck
50 S 3.000 PC 11.34 / 1,000 Truck
1001693343 / 10
80 L 2.000 PC 11.34 / 1,000 Truck
80 M 2.000 PC 11.34 / 1,000 Truck
80 S 3.000 PC 11.34 / 1,000 Truck
80 XS 2.000 PC 11.34 / 1,000 Truck
3000863696 TKT LB 7689 11291868 7ZB1 / LB 7689 14.000 PC 0.14 + 2.0 /- 2.0
1001692775 / 20
60 L 2.000 PC 11.34 / 1,000 Truck
60 S 3.000 PC 11.34 / 1,000 Truck
1001693343 / 20
70 L 2.000 PC 11.34 / 1,000 Truck
70 M 2.000 PC 11.34 / 1,000 Truck
70 S 3.000 PC 11.34 / 1,000 Truck
70 XS 2.000 PC 11.34 / 1,000 Truck
3000865511 TKT LB 7689 11275575 7ZAZ / LB 7689 14.000 PC 0.14 + 2.0 /- 2.0
1001692774 / 10
30 L 2.000 PC 11.34 / 1,000 Truck
30 S 3.000 PC 11.34 / 1,000 Truck
1001693342 / 10


In [29]:
import re
import json

item_pattern = re.compile(
    r"^(\d{10})\s+([A-Z].*?)\s+\d[\d,]*\.\d+\s+PC\b"
)

sales_order_pattern = re.compile(
    r"^(\d{10})\s*/\s*(\d+)\s*$"
)

size_pattern = re.compile(
    r"^(\d+)\s+(\S+)\s+([\d,]+(?:\.\d+)?)\s+PC\b"
)

items = []
current_item = None
current_order = None

for line in clean_text.splitlines():
    line = line.strip()

    # New item
    match = item_pattern.search(line)

    if match:
        current_item = {
            "item_code": match.group(1),
            "item_description": match.group(2),
            "sales_orders": []
        }

        items.append(current_item)
        current_order = None
        continue

    # New sales order
    match = sales_order_pattern.search(line)

    if match and current_item:
        current_order = {
            "sales_order": f"{match.group(1)} / {match.group(2)}",
            "sizes": []
        }

        current_item["sales_orders"].append(current_order)
        continue

    # Size and quantity row
    match = size_pattern.search(line)

    if match and current_order:
        quantity = float(match.group(3).replace(",", ""))

        if quantity.is_integer():
            quantity = int(quantity)

        current_order["sizes"].append({
            "po_line": match.group(1),
            "size": match.group(2),
            "quantity": quantity
        })
        continue

    # Additional item-description line
    if current_item and not current_item["sales_orders"]:
        current_item["item_description"] += " " + line


print(json.dumps(items, indent=4))

[
    {
        "item_code": "3000863694",
        "item_description": "TKT LB 7689 11275526 7YBM / LB 7689",
        "sales_orders": [
            {
                "sales_order": "1001692776 / 20",
                "sizes": [
                    {
                        "po_line": "90",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "90",
                        "size": "S",
                        "quantity": 3
                    }
                ]
            }
        ]
    },
    {
        "item_code": "3000863695",
        "item_description": "TKT LB 7689 11291868 7ZAZ / LB 7689",
        "sales_orders": [
            {
                "sales_order": "1001692775 / 10",
                "sizes": [
                    {
                        "po_line": "50",
                        "size": "L",
                        "quantity": 2
                    },
                 

In [ ]:
[
    {
        "item_code": "3000863694",
        "item_description": "TKT LB 7689 11275526 7YBM / LB 7689",
        "sales_orders": [
            {
                "sales_order": "1001692776 / 20",
                "sizes": [
                    {
                        "po_line": "90",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "90",
                        "size": "S",
                        "quantity": 3
                    }
                ]
            }
        ]
    },
    {
        "item_code": "3000863695",
        "item_description": "TKT LB 7689 11291868 7ZAZ / LB 7689",
        "sales_orders": [
            {
                "sales_order": "1001692775 / 10",
                "sizes": [
                    {
                        "po_line": "50",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "50",
                        "size": "S",
                        "quantity": 3
                    }
                ]
            },
            {
                "sales_order": "1001693343 / 10",
                "sizes": [
                    {
                        "po_line": "80",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "80",
                        "size": "M",
                        "quantity": 2
                    },
                    {
                        "po_line": "80",
                        "size": "S",
                        "quantity": 3
                    },
                    {
                        "po_line": "80",
                        "size": "XS",
                        "quantity": 2
                    }
                ]
            }
        ]
    },
    {
        "item_code": "3000863696",
        "item_description": "TKT LB 7689 11291868 7ZB1 / LB 7689",
        "sales_orders": [
            {
                "sales_order": "1001692775 / 20",
                "sizes": [
                    {
                        "po_line": "60",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "60",
                        "size": "S",
                        "quantity": 3
                    }
                ]
            },
            {
                "sales_order": "1001693343 / 20",
                "sizes": [
                    {
                        "po_line": "70",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "70",
                        "size": "M",
                        "quantity": 2
                    },
                    {
                        "po_line": "70",
                        "size": "S",
                        "quantity": 3
                    },
                    {
                        "po_line": "70",
                        "size": "XS",
                        "quantity": 2
                    }
                ]
            }
        ]
    },
    {
        "item_code": "3000865511",
        "item_description": "TKT LB 7689 11275575 7ZAZ / LB 7689",
        "sales_orders": [
            {
                "sales_order": "1001692774 / 10",
                "sizes": [
                    {
                        "po_line": "30",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "30",
                        "size": "S",
                        "quantity": 3
                    }
                ]
            },
            {
                "sales_order": "1001693342 / 10",
                "sizes": [
                    {
                        "po_line": "40",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "40",
                        "size": "M",
                        "quantity": 2
                    },
                    {
                        "po_line": "40",
                        "size": "S",
                        "quantity": 3
                    },
                    {
                        "po_line": "40",
                        "size": "XS",
                        "quantity": 2
                    }
                ]
            }
        ]
    },
    {
        "item_code": "3000865512",
        "item_description": "TKT LB 7689 11275575 7ZB1 / LB 7689",
        "sales_orders": [
            {
                "sales_order": "1001692774 / 20",
                "sizes": [
                    {
                        "po_line": "10",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "10",
                        "size": "S",
                        "quantity": 3
                    }
                ]
            },
            {
                "sales_order": "1001693342 / 20",
                "sizes": [
                    {
                        "po_line": "20",
                        "size": "L",
                        "quantity": 2
                    },
                    {
                        "po_line": "20",
                        "size": "M",
                        "quantity": 2
                    },
                    {
                        "po_line": "20",
                        "size": "S",
                        "quantity": 3
                    },
                    {
                        "po_line": "20",
                        "size": "XS",
                        "quantity": 2
                    }
                ]
            }
        ]
    }
]